# Curadoria — Pipeline VERDADEIRO (Fontes Oficiais)

Lê todos os CSVs raw do pipeline `pipeline_verdadeiro_fontes_oficiais/raw/`,
processa cada sub-fonte separadamente (Câmara, Senado, STF, Transparência, TSE)
e salva um único CSV curated consolidado com timestamp.

**Sub-fontes processadas:**
- `camara_proposicoes_*` → `texto_principal` = ementa
- `senado_materias_*` → `texto_principal` = Ementa
- `stf_noticias_oficiais_*` → `texto_principal` = titulo + resumo
- `transparencia_contratos_*` → `texto_principal` = objeto do contrato
- `transparencia_emendas_*` → `texto_principal` = frase construída
- `transparencia_despesas_*` → `texto_principal` = frase construída
- `tse_candidatos_2024_*` → `texto_principal` = frase factual (tipo_conteudo = DADO_ELEITORAL)

**Arquivos ignorados nesta curadoria:**
- `tse_candidatos_2024_recursos_*` (catálogo de recursos do TSE, não são dados de candidatos)

## Bibliotecas

In [1]:
import re
import uuid
import pandas as pd

from datetime import datetime
from email.utils import parsedate_to_datetime
from pathlib import Path
from html.parser import HTMLParser

## Configuração de caminhos

In [2]:
NOME_PIPELINE = "pipeline_verdadeiro_fontes_oficiais"

PASTA_RAW     = Path(f"../dados/{NOME_PIPELINE}/raw")
PASTA_CURATED = Path(f"../dados/{NOME_PIPELINE}/curated")
PASTA_CURATED.mkdir(parents=True, exist_ok=True)

print(f"Raw:     {PASTA_RAW}")
print(f"Curated: {PASTA_CURATED}")

Raw:     ..\dados\pipeline_verdadeiro_fontes_oficiais\raw
Curated: ..\dados\pipeline_verdadeiro_fontes_oficiais\curated


## Funções utilitárias

In [3]:
class _StripHTML(HTMLParser):
    def __init__(self):
        super().__init__()
        self._partes = []

    def handle_data(self, data):
        self._partes.append(data)

    def get_text(self):
        return " ".join(self._partes)


def remover_html(texto: str) -> str:
    if not isinstance(texto, str) or not texto.strip():
        return ""
    parser = _StripHTML()
    parser.feed(texto)
    limpo = parser.get_text()
    limpo = re.sub(r"\s+", " ", limpo).strip()
    return limpo


def padronizar_data_iso(valor) -> str:
    # Converte datas ISO ou dd/mm/yyyy para YYYY-MM-DD
    if not isinstance(valor, str) or not valor.strip():
        return ""
    valor = valor.strip()
    formatos = [
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%d/%m/%Y %H:%M:%S",
    ]
    for fmt in formatos:
        try:
            return datetime.strptime(valor, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return ""


def padronizar_data_rfc(valor) -> str:
    # Converte datas RFC 2822 (usadas em feeds RSS/STF) para YYYY-MM-DD.
    # Exemplo: 'Sun, 05 Apr 2026 15:47:00 +0000' → '2026-04-05'
    
    if not isinstance(valor, str) or not valor.strip():
        return ""
    try:
        return parsedate_to_datetime(valor.strip()).strftime("%Y-%m-%d")
    except Exception:
        return padronizar_data_iso(valor)


def normalizar_texto(texto: str) -> str:
    # Remove HTML e normaliza espaços.
    return remover_html(texto)


def gerar_id() -> str:
    return str(uuid.uuid4())


print("Funções utilitárias definidas.")

Funções utilitárias definidas.


## Funções de processamento por sub-fonte

In [4]:
def processar_camara(df_raw: pd.DataFrame) -> pd.DataFrame:
    
    # Câmara dos Deputados — Proposições
    # Colunas relevantes: id, uri, siglaTipo, numero, ano, ementa, dataApresentacao
    df = df_raw.copy()

    df["texto_principal"] = df["ementa"].apply(normalizar_texto)
    df["fonte"]           = "CAMARA_DEPUTADOS"
    df["tipo_conteudo"]   = "PROPOSICAO_LEGISLATIVA"
    df["data_publicacao"] = df["dataApresentacao"].apply(padronizar_data_iso)
    df["url_origem"]      = df["uri"].fillna("").str.strip()

    # Colunas de contexto
    df["tipo_dado"]       = "PROPOSICAO"
    df["orgao_consultado"] = "CAMARA_DEPUTADOS"
    df["contexto_extra"]  = df.apply(
        lambda r: f"{r.get('siglaTipo', '')} {r.get('numero', '')} / {r.get('ano', '')}".strip(),
        axis=1
    )

    return df


def processar_senado(df_raw: pd.DataFrame) -> pd.DataFrame:

    # Senado Federal — Matérias Legislativas
    # Colunas relevantes: Codigo, DescricaoIdentificacao, Sigla, Numero, Ano, Ementa, Autor, Data, UrlDetalheMateria
    df = df_raw.copy()

    df["texto_principal"] = df["Ementa"].apply(normalizar_texto)
    df["fonte"]           = "SENADO_FEDERAL"
    df["tipo_conteudo"]   = "MATERIA_LEGISLATIVA"
    df["data_publicacao"] = df["Data"].apply(padronizar_data_iso)
    df["url_origem"]      = df["UrlDetalheMateria"].fillna("").str.strip()

    df["tipo_dado"]        = "MATERIA"
    df["orgao_consultado"] = "SENADO_FEDERAL"
    df["contexto_extra"]   = df.apply(
        lambda r: f"{r.get('DescricaoIdentificacao', '')} — {r.get('Autor', '')}".strip(" —"),
        axis=1
    )

    return df


def processar_stf(df_raw: pd.DataFrame) -> pd.DataFrame:

    # STF — Notícias Oficiais
    # Colunas relevantes: titulo, resumo, link, data_publicacao, url_feed
    df = df_raw.copy()

    titulo  = df["titulo"].apply(normalizar_texto)
    resumo  = df["resumo"].apply(normalizar_texto)
    df["texto_principal"] = titulo + df.apply(
        lambda r: f" — {remover_html(str(r['resumo']))}" if pd.notna(r["resumo"]) and str(r["resumo"]).strip() not in ("", "nan") else "",
        axis=1
    )
    df["texto_principal"] = df["texto_principal"].apply(lambda t: re.sub(r"\s+", " ", t).strip())

    df["fonte"]           = "STF"
    df["tipo_conteudo"]   = "NOTICIA_OFICIAL"
    df["data_publicacao"] = df["data_publicacao"].apply(padronizar_data_rfc)
    df["url_origem"]      = df["link"].fillna("").str.strip()

    df["tipo_dado"]        = "NOTICIAS_OFICIAIS"
    df["orgao_consultado"] = "STF"
    df["contexto_extra"]   = df["url_feed"].fillna("")

    return df


def processar_contratos(df_raw: pd.DataFrame) -> pd.DataFrame:

    # Portal da Transparência — Contratos Públicos
    # Colunas relevantes: objeto, dataAssinatura, url_consulta, orgao_consultado
    df = df_raw.copy()

    df["texto_principal"] = df["objeto"].apply(normalizar_texto)
    df["fonte"]           = "PORTAL_TRANSPARENCIA"
    df["tipo_conteudo"]   = "CONTRATO_PUBLICO"
    df["data_publicacao"] = df["dataAssinatura"].apply(padronizar_data_iso)
    df["url_origem"]      = df["url_consulta"].fillna("").str.strip()

    df["tipo_dado"]        = "CONTRATOS_PUBLICOS"
    df["orgao_consultado"] = df["orgao_consultado"].fillna("")
    df["contexto_extra"]   = df.apply(
        lambda r: f"Contrato {r.get('numero', '')} — {r.get('orgao_consultado', '')}".strip(" —"),
        axis=1
    )

    return df


def processar_emendas(df_raw: pd.DataFrame) -> pd.DataFrame:

    # Portal da Transparência — Emendas Parlamentares
    # Constrói texto_principal a partir de campos estruturados.
    # Colunas relevantes: tipoEmenda, nomeAutor, localidadeDoGasto, funcao, subfuncao, ano
    df = df_raw.copy()

    def construir_texto(r):
        tipo      = str(r.get("tipoEmenda", "")).strip()
        autor     = str(r.get("nomeAutor", "")).strip()
        localidade = str(r.get("localidadeDoGasto", "")).strip()
        funcao    = str(r.get("funcao", "")).strip()
        subfuncao = str(r.get("subfuncao", "")).strip()
        ano       = str(r.get("ano", "")).strip()
        partes = [p for p in [tipo, "de", autor, "em", localidade, "para", funcao, "/", subfuncao, f"({ano})"] if p and p not in ("nan", "")]
        return " ".join(partes)

    df["texto_principal"] = df.apply(construir_texto, axis=1)
    df["fonte"]           = "PORTAL_TRANSPARENCIA"
    df["tipo_conteudo"]   = "EMENDA_PARLAMENTAR"
    df["data_publicacao"] = df["ano"].apply(
        lambda a: f"{a}-01-01" if isinstance(a, str) and a.strip().isdigit() else ""
    )
    df["url_origem"]      = df["url_consulta"].fillna("").str.strip()

    df["tipo_dado"]        = "EMENDAS_PARLAMENTARES"
    df["orgao_consultado"] = "PORTAL_TRANSPARENCIA"
    df["contexto_extra"]   = df["codigoEmenda"].fillna("").astype(str)

    return df


def processar_despesas(df_raw: pd.DataFrame) -> pd.DataFrame:

    # Portal da Transparência — Despesas Públicas por Órgão
    # Constrói texto_principal a partir de campos financeiros estruturados.
    # Colunas relevantes: orgao, orgaoSuperior, empenhado, ano
    df = df_raw.copy()

    def construir_texto(r):
        orgao   = str(r.get("orgao", "")).strip()
        superior = str(r.get("orgaoSuperior", "")).strip()
        empenhado = str(r.get("empenhado", "")).strip()
        ano     = str(r.get("ano", "")).strip()
        partes = [p for p in [orgao, "vinculado ao", superior, "empenhou", empenhado, "em", ano] if p and p not in ("nan", "")]
        return " ".join(partes)

    df["texto_principal"] = df.apply(construir_texto, axis=1)
    df["fonte"]           = "PORTAL_TRANSPARENCIA"
    df["tipo_conteudo"]   = "DESPESA_PUBLICA"
    df["data_publicacao"] = df["ano"].apply(
        lambda a: f"{a}-01-01" if isinstance(a, str) and a.strip().isdigit() else ""
    )
    df["url_origem"]      = df["url_consulta"].fillna("").str.strip()

    df["tipo_dado"]        = "DESPESAS_PUBLICAS_POR_ORGAO"
    df["orgao_consultado"] = df["orgao_consultado"].fillna("")
    df["contexto_extra"]   = df["orgao"].fillna("")

    return df


def processar_tse(df_raw: pd.DataFrame) -> pd.DataFrame:

    # TSE — Candidatos 2024 (tipo_conteudo = DADO_ELEITORAL)
    # Constrói frase factual sobre o candidato.
    # Preservado para uso futuro como base oficial/evidência.
    # Não deve ser misturado diretamente no dataset final de treino.
    df = df_raw.copy()

    def construir_texto(r):
        nome    = str(r.get("NM_CANDIDATO", "")).strip().title()
        cargo   = str(r.get("DS_CARGO", "")).strip().title()
        partido = str(r.get("NM_PARTIDO", "")).strip()
        uf      = str(r.get("SG_UF", "")).strip()
        municipio = str(r.get("NM_UE", "")).strip().title()
        ano     = str(r.get("ANO_ELEICAO", "")).strip()
        situacao = str(r.get("DS_SIT_TOT_TURNO", "")).strip().lower()
        return (
            f"{nome} concorreu ao cargo de {cargo} em {municipio}/{uf} "
            f"pelo {partido} nas eleições de {ano} e {situacao}."
        )

    df["texto_principal"] = df.apply(construir_texto, axis=1)
    df["fonte"]           = "TSE"
    df["tipo_conteudo"]   = "DADO_ELEITORAL"
    df["data_publicacao"] = df["DT_ELEICAO"].apply(padronizar_data_iso)
    df["url_origem"]      = df["url_arquivo"].fillna("").str.strip()

    df["tipo_dado"]        = "DADOS_ELEITORAIS"
    df["orgao_consultado"] = "TSE"
    df["contexto_extra"]   = df.apply(
        lambda r: f"{r.get('SG_PARTIDO', '')} | {r.get('DS_SITUACAO_CANDIDATURA', '')}",
        axis=1
    )

    return df


print("Funções de processamento por sub-fonte definidas.")

Funções de processamento por sub-fonte definidas.


## Roteamento automático por prefixo de arquivo

In [5]:
# Mapeamento: prefixo do nome do arquivo → função de processamento
ROTEADOR = {
    "camara_proposicoes":                   processar_camara,
    "senado_materias":                       processar_senado,
    "stf_noticias_oficiais":                 processar_stf,
    "transparencia_contratos":               processar_contratos,
    "transparencia_emendas_parlamentares":   processar_emendas,
    "transparencia_despesas_orgaos":         processar_despesas,
    "tse_candidatos_2024_raw":               processar_tse,
}

# Arquivos a ignorar explicitamente
IGNORAR_PREFIXOS = [
    "tse_candidatos_2024_recursos",  # catálogo de recursos, não dados de candidatos
]


def identificar_processador(nome_arquivo: str):
    # Retorna a função de processamento para o arquivo, ou None se deve ser ignorado
    for prefixo in IGNORAR_PREFIXOS:
        if nome_arquivo.startswith(prefixo):
            return None
    for prefixo, func in ROTEADOR.items():
        if nome_arquivo.startswith(prefixo):
            return func
    return None


print("Roteador configurado.")

Roteador configurado.


## Leitura e processamento de todos os CSVs raw

In [6]:
COLUNAS_OBRIGATORIAS = [
    "id_registro",
    "texto_principal",
    "rotulo_preliminar",
    "pipeline",
    "fonte",
    "tipo_conteudo",
    "data_publicacao",
    "url_origem",
    "data_curadoria",
]

COLUNAS_CONTEXTO = [
    "tipo_dado",
    "orgao_consultado",
    "contexto_extra",
    "arquivo_raw_origem",
]

arquivos_raw = sorted(PASTA_RAW.glob("*.csv"))
print(f"Total de arquivos raw encontrados: {len(arquivos_raw)}")

frames_processados = []

for arq in arquivos_raw:
    nome = arq.stem  # nome sem extensão
    processador = identificar_processador(nome)

    if processador is None:
        print(f"  [IGNORADO]    {arq.name}")
        continue

    try:
        df_arq = pd.read_csv(arq, encoding="utf-8-sig", dtype=str)
        df_proc = processador(df_arq)

        df_proc["arquivo_raw_origem"] = arq.name
        df_proc["id_registro"]        = [gerar_id() for _ in range(len(df_proc))]
        df_proc["rotulo_preliminar"]   = "VERDADEIRO"
        df_proc["pipeline"]            = "fontes_oficiais"
        df_proc["data_curadoria"]      = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Garante que as colunas de contexto existam mesmo que não produzidas
        for col in COLUNAS_CONTEXTO:
            if col not in df_proc.columns:
                df_proc[col] = ""

        df_saida = df_proc[COLUNAS_OBRIGATORIAS + COLUNAS_CONTEXTO].copy()
        frames_processados.append(df_saida)
        print(f"  [OK] {arq.name:60s} → {len(df_saida)} registros")

    except Exception as e:
        print(f"  [ERRO] {arq.name}: {e}")

print(f"\nArquivos processados: {len(frames_processados)}")

Total de arquivos raw encontrados: 12
  [OK] camara_proposicoes_raw_2026-05-08_00-54-05.csv               → 20 registros
  [OK] camara_proposicoes_raw_2026-05-09_15-21-47.csv               → 20 registros
  [OK] camara_proposicoes_raw_2026-05-10_02-18-24.csv               → 20 registros
  [OK] senado_materias_raw_2026-05-10_02-28-13.csv                  → 1381 registros
  [OK] stf_noticias_oficiais_raw_2026-05-13_00-15-19.csv            → 10 registros
  [OK] stf_noticias_oficiais_raw_2026-05-16_14-03-48.csv            → 10 registros
  [OK] transparencia_contratos_mec_raw_2026-05-12_23-46-48.csv      → 130 registros
  [OK] transparencia_contratos_mec_raw_2026-05-12_23-51-13.csv      → 2143 registros
  [OK] transparencia_despesas_orgaos_raw_2026-05-13_00-26-55.csv    → 536 registros
  [OK] transparencia_emendas_parlamentares_raw_2026-05-13_00-34-20.csv → 1500 registros
  [OK] tse_candidatos_2024_raw_2026-05-12_23-21-02.csv              → 5834 registros
  [IGNORADO]    tse_candidatos_2024_

## Consolidação e remoção de duplicatas

In [7]:
df_curated = pd.concat(frames_processados, ignore_index=True)
print(f"Total bruto consolidado: {len(df_curated)} registros")

# Remover registros sem texto
antes = len(df_curated)
df_curated = df_curated[df_curated["texto_principal"].str.strip() != ""].copy()
print(f"Removidos sem texto: {antes - len(df_curated)}")

# Remover duplicatas por texto_principal + fonte + url_origem
antes = len(df_curated)
df_curated = df_curated.drop_duplicates(
    subset=["texto_principal", "fonte", "url_origem"],
    keep="first"
).copy()
print(f"Duplicatas removidas: {antes - len(df_curated)}")
print(f"Registros únicos finais: {len(df_curated)}")

Total bruto consolidado: 11604 registros
Removidos sem texto: 0
Duplicatas removidas: 1009
Registros únicos finais: 10595


## Verificação de qualidade

In [8]:
print("=== Verificação de qualidade ===")
print(f"\nShape: {df_curated.shape}")
print(f"\nValores nulos por coluna obrigatória:")
print(df_curated[COLUNAS_OBRIGATORIAS].isnull().sum())
print(f"\nDistribuição por fonte:")
print(df_curated["fonte"].value_counts())
print(f"\nDistribuição por tipo_conteudo:")
print(df_curated["tipo_conteudo"].value_counts())
print(f"\nRegistros com data_publicacao preenchida: {(df_curated['data_publicacao'] != '').sum()}")
df_curated.head(3)

=== Verificação de qualidade ===

Shape: (10595, 13)

Valores nulos por coluna obrigatória:
id_registro          0
texto_principal      0
rotulo_preliminar    0
pipeline             0
fonte                0
tipo_conteudo        0
data_publicacao      0
url_origem           0
data_curadoria       0
dtype: int64

Distribuição por fonte:
fonte
TSE                     5834
PORTAL_TRANSPARENCIA    3330
SENADO_FEDERAL          1381
CAMARA_DEPUTADOS          40
STF                       10
Name: count, dtype: int64

Distribuição por tipo_conteudo:
tipo_conteudo
DADO_ELEITORAL            5834
EMENDA_PARLAMENTAR        1496
MATERIA_LEGISLATIVA       1381
CONTRATO_PUBLICO          1298
DESPESA_PUBLICA            536
PROPOSICAO_LEGISLATIVA      40
NOTICIA_OFICIAL             10
Name: count, dtype: int64

Registros com data_publicacao preenchida: 10555


,id_registro,texto_principal,rotulo_preliminar,pipeline,fonte,tipo_conteudo,data_publicacao,url_origem,data_curadoria,tipo_dado,orgao_consultado,contexto_extra,arquivo_raw_origem
0,6a192a84-b14c-4fe0-bc4f-50de339caa1f,Indica ao Poder Executivo Federal a adoção de ...,VERDADEIRO,fontes_oficiais,CAMARA_DEPUTADOS,PROPOSICAO_LEGISLATIVA,,https://dadosabertos.camara.leg.br/api/v2/prop...,2026-05-17 02:45:06,PROPOSICAO,CAMARA_DEPUTADOS,INC 701 / 2026,camara_proposicoes_raw_2026-05-08_00-54-05.csv
1,cb45b523-7acd-40b5-8822-e63136ccd434,Dispõe sobre a vedação da utilização de dispos...,VERDADEIRO,fontes_oficiais,CAMARA_DEPUTADOS,PROPOSICAO_LEGISLATIVA,,https://dadosabertos.camara.leg.br/api/v2/prop...,2026-05-17 02:45:06,PROPOSICAO,CAMARA_DEPUTADOS,PL 2268 / 2026,camara_proposicoes_raw_2026-05-08_00-54-05.csv
2,17d638ce-7ee9-46ca-88cb-46e7523da765,"Altera a Lei nº 11.959, de 29 de junho de 2009...",VERDADEIRO,fontes_oficiais,CAMARA_DEPUTADOS,PROPOSICAO_LEGISLATIVA,,https://dadosabertos.camara.leg.br/api/v2/prop...,2026-05-17 02:45:06,PROPOSICAO,CAMARA_DEPUTADOS,PL 2266 / 2026,camara_proposicoes_raw_2026-05-08_00-54-05.csv


## Exportação para curated/

In [9]:
data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
caminho_saida = PASTA_CURATED / f"fontes_oficiais_curated_{data_agora}.csv"

df_curated.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Arquivo curated salvo em: {caminho_saida}")
print(f"Total de registros exportados: {len(df_curated)}")
print(f"Data e hora da curadoria: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo curated salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\curated\fontes_oficiais_curated_2026-05-17_02-45-07.csv
Total de registros exportados: 10595
Data e hora da curadoria: 17/05/2026 02:45:08
